# Simultaneous Determination of Microgrid Design & Operation on Multiple Representative Days

A microgrid designer for an industrial park considering going off-grid must simultaneously determine the **equipment capacity** (sizing, investment decision) for PV, batteries, and emergency generators, along with the **charge/discharge and output allocation** (operation decision) across multiple representative days (summer weekday, winter weekday, shoulder season, and weekend).
If the capacity is underestimated, operations will fail on a specific representative day, and if overestimated, investment recovery will deteriorate.

What makes this problem particularly difficult is that the internal resistance loss of the battery is represented by a **hyperbola**: `loss * cap_batt >= k * (charge/discharge output)^2`, and **the design variable of capacity itself appears in the non-linear term**. If capacity and operation are separated, this physical coupling ("the larger it is built, the more efficient it is for the same output, but the higher the investment") disappears completely.

1. **Naive Formulation** — Confirm the location of sizing variables and hyperbolic constraints.
2. **Diagnosis** — Observe with `mk.analyze` and see which symptoms trigger.
3. **Looking Inside the Diagnosis** — Drill down into violations of non-linear constraints in the root LP relaxation and the grounds for triggering numerical scale issues.
4. **Trying Improvements** — Try the recipe for numerical_scale findings and honestly measure its effectiveness.

Subject: `samples/energy_and_microgrid/microgrid_design_operation.py`

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/energy_and_microgrid"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display
from pyscipopt import Model, quicksum

def show(fig):
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import microgrid_design_operation as mdo
from minlpkit.collectors.static_diag import extract_coefficients, scale_summary, residual_scale
from minlpkit.collectors.violation import collect_root_violations, violation_by_type

C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'

def base_layout(title, xtitle, ytitle, height=380):
    ax = dict(gridcolor=C["grid"], linecolor=C["axis"],
              tickfont=dict(color=C["muted"], size=11),
              title_font=dict(color=C["ink2"], size=12), zeroline=False)
    return go.Layout(
        title=dict(text=title, font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
        paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
        font=dict(family=FONT, color=C["ink2"], size=12),
        xaxis=dict(ax, title=xtitle), yaxis=dict(ax, title=ytitle),
        margin=dict(l=60, r=20, t=48, b=48), height=height, hovermode="closest",
        legend=dict(orientation="h", yanchor="bottom", y=1.0, x=1.0, xanchor="right",
                    font=dict(size=11, color=C["ink2"]), bgcolor="rgba(0,0,0,0)"))
print("repo root:", ROOT)

## 1. Naive Formulation

`build_model("default")` is on a scale of 4 representative days $\times$ 14 hours. Design variables (`cap_pv`, `cap_batt` are continuous, `n_gen` is integer) are shared across all representative days, while operation variables are held independently for each representative day $\times$ hour.

In [ ]:
m = mdo.build_model("default")
nDay, nT = m.data["dims"]
n_int = sum(1 for v in m.getVars() if v.vtype() in ("INTEGER", "BINARY"))
n_cont = sum(1 for v in m.getVars() if v.vtype() == "CONTINUOUS")
n_nonlin = sum(1 for c in m.getConss() if not c.isLinear())
print(f"代表日={nDay} 時刻={nT}")
print(f"変数: 整数(発電機台数) {n_int} 個 / 連続(サイジング+運用) {n_cont} 個")
print(f"制約: {m.getNConss()} 本(内 非線形 = {n_nonlin} 本 = 代表日×時刻ぶんの蓄電池損失式)")
print(f"BATT_K_LOSS={mdo.BATT_K_LOSS}  蓄電池容量範囲=[{5.0},{1500.0}]")
print("非線形制約の例(1本):", "loss*cap_batt >= BATT_K_LOSS*(p_charge+p_discharge)^2")

## 2. Diagnosis (`mk.analyze`)

In [ ]:
report = mk.analyze(lambda: mdo.build_model("default"), name="microgrid-default", time_limit=30)
print(report.summary())
print()
print({k: report.metrics.get(k) for k in [
    "gap", "nodes", "nsols", "spatial_share", "has_nonlinear",
    "bottleneck_type", "bottleneck_rel_viol",
    "residual_coef_ratio", "coef_ratio", "residual_bigm_count"]})

Despite `has_nonlinear=True` (a true non-convex MINLP with hyperbolic constraints), `spatial_share=0` and `nodes=1` — it is solved at the root without using any spatial branching. The only issue triggered in the diagnosis is **numerical scale** (`numerical_scale`), and the non-convexity itself does not appear as a symptom. In Section 3, we delve into the reason for this (why this non-linear constraint is "not difficult") as well as the basis for the numerical scale trigger.

## 3. Looking Inside the Diagnosis

### 3-1. Why is Spatial Branching Zero Despite Being "Non-Convex"? — Looking at Root LP Relaxation Violations

`collect_root_violations` substitutes the root LP relaxation solution into the non-linear constraints to measure the actual violation amount. If the non-linear term is truly "ineffective", the violation should be close to zero.

In [ ]:
m3 = mdo.build_model("default")
vdf = collect_root_violations(m3)
by_type = violation_by_type(vdf)
print(by_type.head(8).to_string(index=False))

The `batt_loss_*` constraints have a relative violation of 10-35% on average in the root LP relaxation — the non-linear terms are far from "ineffective"; they deviate significantly from the linear relaxation. Yet, the search tree finishes at the root node because **this constraint is not non-convex but convex**.

`loss * cap_batt >= k * p^2` (where `p = p_charge + p_discharge`, `cap_batt > 0`) is equivalent to `loss >= k * p^2 / cap_batt`, which is the lower epigraph of the **perspective function** of the convex quadratic function `k*p^2`. Since the perspective transformation preserves convexity, as long as `cap_batt` is bounded by a lower limit of 5 (>0), this constraint is **jointly convex** as a function of `(p, cap_batt, loss)`. SCIP can tighten this kind of quadratic constraint (equivalent to a rotated second-order cone) using linear approximation cuts (outer approximation) without requiring integer $\times$ continuous spatial branching — showing that the strength of the coupling where "the design variable appears in the non-linear term" and "whether that non-linearity makes the search difficult" are on different axes. (This is also why `minlpkit.perspective_quadratic` provides the same transformation as a semi-continuous on/off version.)

### 3-2. Numerical Scale — Grounds for Triggering `residual_coef_ratio`

The monetary scale of the investment cost `COST_GEN_UNIT=55000` [$] for `n_gen`, which is orders of magnitude different from other physical quantities (kW, ratios), pushes up the `residual_coef_ratio`. Let's actually check the source of the minimum and maximum after presolve.

In [ ]:
m4 = mdo.build_model("default")
m4.hideOutput()
pre = residual_scale(m4)
print(f"presolve後: min={pre['min']:.3g} max={pre['max']:.3g} ratio={pre['ratio']:.3g}")

m5 = mdo.build_model("default")
m5.hideOutput(); m5.presolve()
df_coef = extract_coefficients(m5)
df_sorted = df_coef.sort_values("magnitude")
print("--- 最小側 ---")
print(df_sorted.head(5)[["source", "magnitude", "name"]].to_string(index=False))
print("--- 最大側 ---")
print(df_sorted.tail(5)[["source", "magnitude", "name"]].to_string(index=False))

In [ ]:
fig = go.Figure(layout=base_layout(
    "presolve後の係数分布(出所別) — 最大は発電機投資費(55,000$/台)、最小は物理量(比率・kW)",
    "出所", "|係数| (log scale)", height=400))
for src, color in [("制約係数", C["s1"]), ("RHS/LHS", C["muted"]), ("目的係数", C["warn"]),
                   ("変数境界", C["s2"])]:
    sub = df_coef[df_coef["source"] == src]
    if sub.empty:
        continue
    fig.add_trace(go.Box(y=sub["magnitude"], name=src, marker_color=color, boxpoints="outliers"))
fig.update_yaxes(type="log")
show(fig)

The minimum side is the coefficient of the `pv_cap_*` constraints ($\approx 0.17$, which is the PV output ratio itself for that representative day/hour — the solar irradiance profile per unit capacity), while the maximum side is the objective coefficient for the investment cost of `n_gen` (55,000$).
Neither of these is a rounding error from presolve but **values inherent to the model**. The reality of the `residual_coef_ratio` is that **quantities with different unit systems — "investment cost (dollars)" and "PV output ratio (dimensionless, 0-1)" — coexist in the same coefficient matrix**.
Unlike the "looseness of Big-M" focused on by `static_diag.matrix_condition`, this is a **numerical scale structurally inherent to this type of investment $\times$ operation model due to the order of magnitude difference between currency units and physical units**. We will actually try to fix this in Section 4.

## 4. Trying Improvements — Scaling the Objective Function ($ $\rightarrow$ $k)

The recipe for `numerical_scale` is "Scaling, Indicator/SOS of Big-M, and coefficient reformulation". Since the main cause here is the difference in unit orders of magnitude rather than Big-M, we create a version where the investment cost is converted to thousands of dollars ($k) (the optimal solution and optimal investment/operation decisions remain unchanged, only the value of the objective function is scaled), and compare both the numerical scale indicators and actual solving performance.

In [ ]:
def build_scaled(scale="default"):
    d = mdo._data(scale)
    nDay, nT = d["nDay"], d["nT"]
    pv_profile, demand, day_weight = d["pv_profile"], d["demand"], d["day_weight"]
    K = 1000.0  # 投資費を$→$kへ(残りは物理量のままなので係数レンジが縮む)

    m = Model("Microgrid_Design_Operation_Scaled")
    DAY, T = range(nDay), range(nT)
    cap_pv = m.addVar(vtype="C", lb=0.0, ub=1200.0, name="cap_pv")
    cap_batt = m.addVar(vtype="C", lb=5.0, ub=1500.0, name="cap_batt")
    n_gen = m.addVar(vtype="I", lb=0, ub=mdo.N_GEN_MAX, name="n_gen")
    p_pv, p_gen, p_charge, p_discharge, p_grid, soc, loss = ({} for _ in range(7))
    z_gen = {}
    for dday in DAY:
        for t in T:
            p_pv[dday, t] = m.addVar(vtype="C", lb=0.0, name=f"p_pv_{dday}_{t}")
            p_gen[dday, t] = m.addVar(vtype="C", lb=0.0, name=f"p_gen_{dday}_{t}")
            z_gen[dday, t] = m.addVar(vtype="I", lb=0, ub=mdo.N_GEN_MAX, name=f"z_gen_{dday}_{t}")
            p_charge[dday, t] = m.addVar(vtype="C", lb=0.0, name=f"p_charge_{dday}_{t}")
            p_discharge[dday, t] = m.addVar(vtype="C", lb=0.0, name=f"p_discharge_{dday}_{t}")
            p_grid[dday, t] = m.addVar(vtype="C", lb=0.0, name=f"p_grid_{dday}_{t}")
            loss[dday, t] = m.addVar(vtype="C", lb=0.0, name=f"loss_{dday}_{t}")
        for t in range(nT + 1):
            soc[dday, t] = m.addVar(vtype="C", lb=0.0, name=f"soc_{dday}_{t}")
    for dday in DAY:
        m.addCons(soc[dday, 0] == 0.5 * cap_batt)
        m.addCons(soc[dday, nT] >= 0.5 * cap_batt)
        for t in T:
            m.addCons(p_pv[dday, t] <= cap_pv * float(pv_profile[dday, t]))
            m.addCons(z_gen[dday, t] <= n_gen)
            m.addCons(p_gen[dday, t] <= mdo.GEN_UNIT_CAP * z_gen[dday, t])
            m.addCons(p_charge[dday, t] <= mdo.MAX_CRATE * cap_batt)
            m.addCons(p_discharge[dday, t] <= mdo.MAX_CRATE * cap_batt)
            m.addCons(soc[dday, t] <= cap_batt)
            m.addCons(loss[dday, t] * cap_batt >= mdo.BATT_K_LOSS
                      * (p_charge[dday, t] + p_discharge[dday, t])
                      * (p_charge[dday, t] + p_discharge[dday, t]))
            m.addCons(soc[dday, t + 1] == soc[dday, t] + mdo.ETA_C * p_charge[dday, t]
                      - p_discharge[dday, t] / mdo.ETA_D - loss[dday, t])
            m.addCons(p_pv[dday, t] + p_gen[dday, t] + p_discharge[dday, t] + p_grid[dday, t]
                      == float(demand[dday, t]) + p_charge[dday, t])
    capex_k = (mdo.COST_PV / K) * cap_pv + (mdo.COST_BATT / K) * cap_batt \
        + (mdo.COST_GEN_UNIT / K) * n_gen
    opex_k = quicksum(float(day_weight[dday]) / K * (
        mdo.FUEL_COST * p_gen[dday, t] + mdo.GRID_COST * p_grid[dday, t]
    ) for dday in DAY for t in T)
    m.setObjective(capex_k + opex_k, "minimize")
    m.data = dict(cap_pv=cap_pv, cap_batt=cap_batt, n_gen=n_gen)
    return m

pre_scale = scale_summary(extract_coefficients(mdo.build_model("default")))
post_scale = scale_summary(extract_coefficients(build_scaled("default")))
print(f"presolve前(元): ratio={pre_scale['ratio']:.3g}  max={pre_scale['max']:.3g}")
print(f"presolve前($k化): ratio={post_scale['ratio']:.3g}  max={post_scale['max']:.3g}")

In [ ]:
df = mk.compare_variants(
    {"元($単位)": lambda: mdo.build_model("default"),
     "$k単位": build_scaled}, time_limit=30)
df[["variant", "root_dual", "final_dual", "final_gap", "nodes"]]

In [ ]:
orig_row = df.loc[df["variant"] == "元($単位)"].iloc[0]
scaled_row = df.loc[df["variant"] == "$k単位"].iloc[0]
fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.12,
    subplot_titles=("係数レンジ比(静的、presolve前)", "最終gap [%]", "探索ノード数"))
labels = ["元($単位)", "$k単位"]
bar_colors = [C["muted"], C["s1"]]
fig.add_trace(go.Bar(x=labels, y=[pre_scale["ratio"], post_scale["ratio"]],
    marker_color=bar_colors, text=[f"{v:.2e}" for v in [pre_scale["ratio"], post_scale["ratio"]]],
    textposition="outside", cliponaxis=False, showlegend=False), row=1, col=1)
fig.update_yaxes(type="log", row=1, col=1)
fig.add_trace(go.Bar(x=labels, y=[orig_row["final_gap"] * 100, scaled_row["final_gap"] * 100],
    marker_color=bar_colors, text=[f"{v*100:.2f}%" for v in [orig_row["final_gap"], scaled_row["final_gap"]]],
    textposition="outside", cliponaxis=False, showlegend=False), row=1, col=2)
fig.add_trace(go.Bar(x=labels, y=[orig_row["nodes"], scaled_row["nodes"]],
    marker_color=bar_colors, text=[int(v) for v in [orig_row["nodes"], scaled_row["nodes"]]],
    textposition="outside", cliponaxis=False, showlegend=False), row=1, col=3)
fig.update_layout(paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), height=380,
    margin=dict(l=40, r=20, t=64, b=40),
    title=dict(text="目的関数の単位スケール変換の効果 — 係数レンジは大きく縮むが求解性能は元々十分速い",
               x=0.01, font=dict(color=C["ink"], size=15)))
fig.update_yaxes(gridcolor=C["grid"], linecolor=C["axis"], zeroline=False)
show(fig)

The coefficient range ratio shrinks significantly with the conversion to `$k` (because the maximum value of the investment cost becomes 1/1000). On the other hand, `root_dual`/`final_gap`/`nodes` are almost the same (already at a gap of about 0.1% at root node 1) —— **at this scale, the effect of unit conversion is limited to "cleaning up diagnostic indicators" and does not appear in actual solving performance**. This is consistent with the conclusion in Section 3 that "non-linear terms are not difficult because they are convex", and aligns with the structure seen in `08_condition_number.ipynb` where "small problems can be solved by presolve alone, making the effect of numerical soundness unmeasurable".

## Summary

- The core of this problem is the simultaneous determination of "PV/Battery/Generator capacity (investment)" and "operation across multiple representative days," where the battery loss hyperbola `loss*cap_batt >= k*p^2` involves the design variable of capacity in the non-linear term — a true physical coupling that disappears if design and operation are separated.
- However, **this hyperbola is mathematically convex** (the perspective transformation of a quadratic function), and does not require spatial branching as long as the lower bound of `cap_batt` is positive. Even with up to a 35% violation in the root LP relaxation, SCIP can tighten it purely with outer approximation cuts. We empirically demonstrated that the strength of coupling "where design variables appear in non-linear terms" is independent of "whether that non-linearity makes the search difficult."
- The triggered `numerical_scale` is not due to Big-M but the "order of magnitude difference between units of investment cost (dollars) and operation volume (kW, ratio)". Converting to $k units shrinks the coefficient range, but at this scale, the solving performance itself was already sufficient, so the effect was unmeasurable (an honest result).

### Business Decisions Corresponding to this Diagnosis

When a microgrid designer determines the scale of investment, solving while maintaining this non-linear coupling is necessary to correctly capture the trade-off that "increasing capacity improves operational efficiency but increases investment." The diagnosis in this notebook provides both reassurance that "this non-linearity (being convex) does not hinder proof of optimality" and practical prioritization that "correcting numerical scale is an organizational cleanup for diagnosis, but does not affect the solving speed at this scale."

Related: [Method Guide 8. Condition Number and Numerical Soundness](../../playbook/08-condition.md) /
Sample Catalog [Energy and Microgrid](../../samples/energy_and_microgrid.md)